# Dodatni seedovi za Korake 1 i 2

Dopuna merenja iz `dopuna_a100_sve.ipynb` sa sedam novih seedova, radi uparenog testa
statisticke znacajnosti. Sa tri izvrsavanja moglo se pokazati samo da li se smer razlike
drzi, a sa deset se razlika moze i kvantifikovati.

Protokol je namerno **identican** postojecim merenjima, jer se rezultati objedinjuju sa
seedovima 42, 123 i 7. Nista se ne menja osim broja seedova, uz jedan dodatak: upisuje se
i epoha u kojoj je postignuta najbolja validaciona tacnost, sto postojeca merenja ne beleze.
Bez te kolone se ne moze proveriti da li je trening imao dovoljno epoha.

Korak 4 se ne ponavlja. Njegova devijacija preko tri izvrsavanja je najvise 0,53 procentna
poena, pa dodatni seedovi tu ne bi nista promenili, a graf ogbn-products bi trebalo iznova
skidati.

Rezultati se upisuju u `results_a100/run_<SEED>/`, isto gde su i postojeci, pa ih
`agregacija_a100.py` sam pronalazi. Vec izmereno se preskace, pa se notebook moze
pustiti iznova posle prekida.

Procena trajanja je oko 34 minuta po seedu, dakle oko cetiri sata za svih sedam.

In [ ]:
SEEDOVI = [1, 2, 3, 4, 5, 6, 8]

In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader, ClusterData, ClusterLoader
from torch_geometric.utils import to_undirected, degree
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}, '
         f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
log.info(f'jezgara {os.cpu_count()}, deljena memorija {shutil.disk_usage("/dev/shm").total/1e9:.0f} GB')


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = Path(f'rezultati/run_{seed}')
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(seed, fajl, red):
    p = direktorijum(seed) / fajl
    novi = not p.exists()
    with open(p, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(seed, fajl, kolona):
    p = direktorijum(seed) / fajl
    if not p.exists():
        return set()
    with open(p) as f:
        return {r[kolona] for r in csv.DictReader(f)}

## Podesavanja i model

Sve vrednosti su prepisane iz `dopuna_a100_sve.ipynb` bez izmene, jer se rezultati
objedinjuju sa postojecim seedovima.

In [ ]:
SKRIVENI, LR = 256, 0.01
EPOHA, BATCH, OCENA_SVAKIH = 50, 512, 5
UCITAVANJE = dict(num_workers=8, persistent_workers=True, prefetch_factor=4, pin_memory=True)

K2_KORAKA, K2_SLOJEVA, K2_BATCH_KLASTERA, K2_OCENA_SVAKIH = 5000, 3, 20, 250
KLASTERI = [100, 500, 1000, 2000]

KORAK1 = [
    ('L2_mali',       2, [5, 5]),
    ('L2_standardni', 2, [15, 10]),
    ('L2_veliki',     2, [25, 20]),
    ('L3_mali',       3, [5, 5, 5]),
    ('L3_standardni', 3, [15, 10, 5]),
    ('L3_veliki',     3, [25, 15, 10]),
    ('L4_mali',       4, [5, 5, 5, 5]),
    ('L4_standardni', 4, [15, 10, 5, 5]),
    ('L4_veliki',     4, [25, 15, 10, 5]),
]


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt, ceo_podgraf=False):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    if ceo_podgraf:
        maska = batch.train_mask
        if maska.sum() == 0:
            return
        gubitak = F.cross_entropy(out[maska], batch.y[maska])
    else:
        maska = batch.train_mask[:batch.batch_size]
        if maska.sum() == 0:
            return
        gubitak = F.cross_entropy(out[:batch.batch_size][maska],
                                  batch.y[:batch.batch_size][maska])
    gubitak.backward()
    opt.step()

## Ucitavanje grafa

Ucitava se jednom za svih sedam seedova. Citatna mreza je usmerena pa se pretvara u neusmerenu.

In [ ]:
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name='ogbn-arxiv')

potega_usmereno = data.num_edges
data.edge_index = to_undirected(data.edge_index)
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True

ulazni = degree(data.edge_index[1], num_nodes=data.num_nodes)
log.info(f'ogbn-arxiv: {data.num_nodes:,} cvorova, {data.num_edges:,} potega '
         f'(usmereno je bilo {potega_usmereno:,})')
log.info(f'ulazni stepen: prosek {ulazni.mean():.2f}, medijana {ulazni.median():.0f}, '
         f'najveci {ulazni.max():.0f}')


@torch.no_grad()
def oceni(model):
    model.eval()
    x = data.x.to(device, non_blocking=True)
    ei = data.edge_index.to(device, non_blocking=True)
    out = model(x, ei).argmax(dim=-1, keepdim=True).cpu()
    del x, ei
    torch.cuda.empty_cache()
    return {s: evaluator.eval({'y_true': data.y[split_idx[s]].unsqueeze(1),
                               'y_pred': out[split_idx[s]]})['acc']
            for s in ['train', 'valid', 'test']}

## Korak 1: geometrija node-wise uzorkovanja

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'korak1_graphsage.csv', 'oznaka')
    for oznaka, slojeva, S in KORAK1:
        if oznaka in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('-' * 70)
        log.info(f'seed {SEED} | Korak 1 | {oznaka}: L={slojeva}, S={S}, B={BATCH}')

        loader = NeighborLoader(data, num_neighbors=S, input_nodes=split_idx['train'],
                                batch_size=BATCH, shuffle=True, **UCITAVANJE)
        model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, slojeva).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, naj_epoha, vrh, vremena = 0.0, float('nan'), 0, 0.0, []
        for ep in range(EPOHA):
            model.train()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            t0 = time.time()
            for batch in loader:
                trening_korak(model, batch, opt)
            torch.cuda.synchronize()
            vremena.append(time.time() - t0)
            vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
            if (ep + 1) % OCENA_SVAKIH == 0 or ep == 0:
                a = oceni(model)
                if a['valid'] > naj_val:
                    naj_val, naj_test, naj_epoha = a['valid'], a['test'], ep + 1
                log.info(f'  epoha {ep+1:3d}: val={a["valid"]*100:5.2f}% '
                         f'test={a["test"]*100:5.2f}% {vremena[-1]:5.2f}s {vrh:6.0f}MB')

        upisi_red(SEED, 'korak1_graphsage.csv', {
            'familija': 'GraphSAGE', 'oznaka': oznaka, 'slojeva': slojeva, 'S': str(S),
            'batch': BATCH, 'koraka_po_epohi': len(loader), 'ukupno_koraka': len(loader) * EPOHA,
            'naj_val': naj_val, 'naj_test': naj_test, 'naj_epoha': naj_epoha,
            'vreme_po_epohi': float(np.mean(vremena)),
            'vreme_po_koraku_ms': float(np.mean(vremena)) / len(loader) * 1000,
            'memorija_mb': vrh})
        log.info(f'seed {SEED} | {oznaka}: test={naj_test*100:.2f}% (epoha {naj_epoha}), '
                 f'{np.mean(vremena):.2f}s/epohi, {vrh:.0f}MB')

        del model, opt, loader
        gc.collect()
        torch.cuda.empty_cache()

log.info('Korak 1 zavrsen za sve seedove')

## Korak 2: subgraph-wise uzorkovanje

Trening ide na fiksnom broju gradijentnih koraka, istom za sve vrednosti K, jer broj
koraka po epohi zavisi od broja klastera.

In [ ]:
for SEED in SEEDOVI:
    uradjeno = vec_uradjeno(SEED, 'korak2_clustergcn.csv', 'oznaka')
    for K in KLASTERI:
        if f'K{K}' in uradjeno:
            continue
        postavi_seed(SEED)
        log.info('-' * 70)
        log.info(f'seed {SEED} | Korak 2 | K={K}, {K2_KORAKA} koraka')

        t0 = time.time()
        cluster_data = ClusterData(data, num_parts=K, recursive=False)
        metis_vreme = time.time() - t0
        log.info(f'  METIS: {metis_vreme:.1f}s')

        loader = ClusterLoader(cluster_data, batch_size=K2_BATCH_KLASTERA, shuffle=True,
                               num_workers=UCITAVANJE['num_workers'], persistent_workers=True)
        model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, K2_SLOJEVA).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)

        naj_val, naj_test, naj_korak, vrh = 0.0, float('nan'), 0, 0.0
        trening_vreme, korak, epoha = 0.0, 0, 0
        torch.cuda.reset_peak_memory_stats()
        while korak < K2_KORAKA:
            epoha += 1
            model.train()
            for batch in loader:
                torch.cuda.synchronize()
                t0 = time.time()
                trening_korak(model, batch, opt, ceo_podgraf=True)
                torch.cuda.synchronize()
                vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
                trening_vreme += time.time() - t0
                korak += 1
                if korak % K2_OCENA_SVAKIH == 0:
                    a = oceni(model)
                    if a['valid'] > naj_val:
                        naj_val, naj_test, naj_korak = a['valid'], a['test'], korak
                    model.train()
                    torch.cuda.reset_peak_memory_stats()
                if korak >= K2_KORAKA:
                    break

        upisi_red(SEED, 'korak2_clustergcn.csv', {
            'familija': 'ClusterGCN', 'oznaka': f'K{K}', 'slojeva': K2_SLOJEVA, 'klastera': K,
            'batch_klastera': K2_BATCH_KLASTERA, 'koraka_po_epohi': len(loader),
            'ukupno_koraka': korak, 'epoha': epoha, 'metis_s': metis_vreme,
            'naj_val': naj_val, 'naj_test': naj_test, 'naj_korak': naj_korak,
            'vreme_po_koraku_ms': trening_vreme / korak * 1000, 'memorija_mb': vrh})
        log.info(f'seed {SEED} | K={K}: test={naj_test*100:.2f}% (korak {naj_korak}), '
                 f'METIS={metis_vreme:.1f}s, {trening_vreme/korak*1000:.1f} ms/korak, {vrh:.0f}MB')

        del model, opt, loader, cluster_data
        gc.collect()
        torch.cuda.empty_cache()

log.info('Korak 2 zavrsen za sve seedove')

## Sazetak po seedu

In [ ]:
for SEED in SEEDOVI:
    d = direktorijum(SEED)
    json.dump({'seed': SEED, 'datum': datetime.now().isoformat(),
               'platforma': 'nacionalna platforma za vestacku inteligenciju',
               'kartica': torch.cuda.get_device_name(0),
               'compute_capability': f'{cc[0]}.{cc[1]}',
               'jezgara': os.cpu_count(),
               'deljena_memorija_gb': round(shutil.disk_usage('/dev/shm').total / 1e9, 1),
               'graf_neusmeren': True,
               'epoha_korak1': EPOHA, 'batch_korak1': BATCH,
               'koraka_korak2': K2_KORAKA, 'batch_klastera': K2_BATCH_KLASTERA,
               'skriveni': SKRIVENI, 'lr': LR},
              open(d / 'sazetak.json', 'w'), indent=2)
    log.info(f'--- seed {SEED} ---')
    for fajl, kljuc in [('korak1_graphsage.csv', 'oznaka'), ('korak2_clustergcn.csv', 'oznaka')]:
        p = d / fajl
        if not p.exists():
            continue
        for r in csv.DictReader(open(p)):
            log.info(f"  {r[kljuc]:15s} test={float(r['naj_test'])*100:5.2f}%  "
                     f"{float(r['memorija_mb']):7.0f}MB")

log.info('Sve zavrseno')